In [1]:
import sys
print(sys.executable)

/Users/shirley/Documents/AI/Projects/product-voice-analytics/.venv/bin/python


# Notebook 03 — DistilBERT Fine-tuning (KEY)

Fine-tune DistilBERT on the 100K sample for 3-class sentiment classification.

Labels: 0 = negative, 1 = neutral, 2 = positive  
Device: MPS (Apple Silicon) → CUDA → CPU  
Output: saved model + tokenizer to `models/distilbert/`

In [4]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(os.getcwd()) 

/Users/shirley/Documents/AI/Projects/product-voice-analytics


In [5]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from src.config import (
    PROCESSED_SAMPLE_PATH,
    DISTILBERT_PATH,
    DISTILBERT_MAX_LEN,
    DISTILBERT_BATCH_SIZE,
    RANDOM_SEED,
    TEST_SIZE,
)

## 1. Config

In [6]:
EPOCHS    = 5
LR        = 2e-5
N_LABELS  = 3
MODEL_NAME = 'distilbert-base-uncased'

def get_device():
    if torch.backends.mps.is_available():
        return torch.device('mps')
    elif torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

device = get_device()
print(f'Using device: {device}')

Using device: mps


## 2. Load Data

In [16]:
df = pd.read_csv(PROCESSED_SAMPLE_PATH)

df['clean_text'] = df['clean_text'].fillna('').astype(str)
# drop any rows where clean_text is empty — can happen if reviewText was malformed
df = df[df['clean_text'].str.strip().astype(bool)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df['label']
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_SEED,
    stratify=train_df['label']
)


# smoke test — remove before full run
train_df = train_df.sample(1000, random_state=42)
val_df   = val_df.sample(200, random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(train_df['label'].value_counts().sort_index())

Train: 1000 | Val: 200 | Test: 19973
label
0    210
1    104
2    686
Name: count, dtype: int64


## 3. Dataset

In [17]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = ReviewDataset(train_df['clean_text'].tolist(), train_df['label'].tolist(), tokenizer, DISTILBERT_MAX_LEN)
val_dataset   = ReviewDataset(val_df['clean_text'].tolist(),   val_df['label'].tolist(),   tokenizer, DISTILBERT_MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=DISTILBERT_BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=DISTILBERT_BATCH_SIZE)

## 4. Model, Optimizer, Scheduler

In [18]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=N_LABELS)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=LR)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(0.1 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 5. Training Loop

In [19]:
train_losses = []
val_losses   = []

for epoch in range(EPOCHS):

    # --- train ---
    model.train()
    epoch_train_loss = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- validate ---
    model.eval()
    epoch_val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            epoch_val_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()
            val_preds.extend(preds)
            val_labels.extend(labels.cpu().tolist())

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    val_acc = accuracy_score(val_labels, val_preds)
    val_f1  = f1_score(val_labels, val_preds, average='macro')

    print(f'Epoch {epoch+1}/{EPOCHS} | train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f} | val_acc: {val_acc:.4f} | val_f1: {val_f1:.4f}')

KeyboardInterrupt: 

## 6. Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Train loss')
plt.plot(range(1, EPOCHS + 1), val_losses,   label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('DistilBERT Fine-tuning — Loss Curve')
plt.legend()
plt.tight_layout()
plt.savefig('models/loss_curve.png', dpi=150)
plt.show()

## 7. Save Model

In [ ]:
os.makedirs(DISTILBERT_PATH, exist_ok=True)
model.save_pretrained(DISTILBERT_PATH)
tokenizer.save_pretrained(DISTILBERT_PATH)
print(f'Model saved to {DISTILBERT_PATH}')